In [1]:
# Solution of a multi-product supply chain (SC) problem in Julia
# We now let the model decide where to install technologies
using JuMP, HiGHS

# sets 
P = ["DM"]
S = ["DF"] 
D = ["CF", "SF"]
L = ["DM->CF","DM->SF"]

# supply data
sprod = ["DM"]
sprod = Dict(zip(S,sprod))
sub = [1000] # ton
sub = Dict(zip(S,sub))
sbid = [0] # $/ton
sbid = Dict(zip(S,sbid))

#demand data
dprod = ["DM","DM"]
dprod = Dict(zip(D,dprod))
dub = [500,500] # ton 
dub = Dict(zip(D,dub))
dbid = [0.5,1.5] # $/ton
dbid = Dict(zip(D,dbid))

# transport data
flocs = ["DF","DF"]
flocr = ["CF","SF"]
flocs = Dict(zip(L,flocs))
flocr = Dict(zip(L,flocr))
fub = [1000,1000] # ton
fub = Dict(zip(L,fub))
fprod = ["DM","DM"]
fprod = Dict(zip(L,fprod))
fbid = [0.1,0.2] # $/ton
fbid = Dict(zip(L,fbid))

Dict{String, Float64} with 2 entries:
  "DM->CF" => 0.1
  "DM->SF" => 0.2

In [2]:
P,S,D

(["DM"], ["DF"], ["CF", "SF"])

In [3]:
using JuMP, HiGHS

# define model object
m = Model(HiGHS.Optimizer); 

# variables
@variable(m, s[S]>=0)
@variable(m, d[D]>=0)
@variable(m, f[L]>=0)

# capacity constraints
@constraint(m, [i in S], s[i] <= sub[i])
@constraint(m, [j in D], d[j] <= dub[j])
@constraint(m, [l in L], f[l] <= fub[l])

# supply balance constraints
@constraint(m, sbal[i in S],  s[i]  ==  sum(f[l] for l in L if flocs[l] == i && fprod[l] == sprod[i]))

# demand balance constraints
@constraint(m, dbal[j in D],  d[j]  ==  sum(f[l] for l in L if flocr[l] == j && fprod[l] == dprod[j]))

#objective function  
dcost = @expression(m, sum(dbid[j]*d[j] for j in D))
scost = @expression(m, sum(sbid[i]*s[i] for i in S))
fcost = @expression(m, sum(fbid[l]*f[l] for l in L))

@objective(m, Max, dcost - scost - fcost)

print(m)

In [4]:
optimize!(m)


Presolving model
0 rows, 0 cols, 0 nonzeros
0 rows, 0 cols, 0 nonzeros
Presolve : Reductions: rows 0(-8); columns 0(-5); elements 0(-12) - Reduced to empty
Solving the original LP from the solution after postsolve
Model   status      : Optimal
Objective value     :  8.5000000000e+02
HiGHS run time      :          0.00


In [5]:
# get results for all suppliers

# allocations
sa = zeros(length(S))
sa = Dict(zip(S,sa))
for i in S
    sa[i]=value.(s)[i]
    print("sa[",i,"] = ",sa[i]," unit \n")
end


sa[DF] = 1000.0 unit 


In [6]:
# get results for all consumers

da = zeros(length(D))
da = Dict(zip(D,da))
for i in D
    da[i]=value.(d)[i]    
    print("da[",i,"] = ",da[i]," unit \n")
end


da[CF] = 500.0 unit 
da[SF] = 500.0 unit 


In [7]:
# get results for all transporters

fa = zeros(length(L))
fa = Dict(zip(L,fa))
for l in L
    fa[l]=value.(f)[l]    
    print("fa[",l,"] = ",fa[l]," unit \n")
end

fa[DM->CF] = 500.0 unit 
fa[DM->SF] = 500.0 unit 
